In [3]:
import numpy as np
from loguru import logger

from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from robot_imitation_glue.base import BaseEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from robot_imitation_glue.ur5station.ur5_robot_env import abs_joint_policy_action_to_joint_pose

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
root_dir="/storage/rproesma/clothes-hanger/datasets/clothes-hanger-v3-raw"
repo_id="tmp"

dataset = LeRobotDataset(repo_id=repo_id, root=root_dir)

Resolving data files:   0%|          | 0/180 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def replay_episode(
    dataset: LeRobotDataset,
    action_to_env_converter,
    image_key,
    dataset_image_key,
    episode_idx: int = 0,
    fps=10,
):
    logger.debug("Predict episode")
    episode_indices = dataset.episode_data_index
    logger.debug(f"episode_indices = {episode_indices}")
    episode_start_idx = episode_indices["from"][episode_idx].item()
    logger.debug(f"episode_start_idx = {episode_start_idx}")
    episode_to_idx = episode_indices["to"][episode_idx].item()
    logger.debug(f"episode_to_idx = {episode_to_idx}")
    logger.debug(f'dataset keys: {list(dataset[episode_start_idx].keys())}')
    dataset_initial_image = dataset[episode_start_idx][image_key]
    logger.debug(f"dataset_initial_image.shape = {dataset_initial_image.shape}")


    # convert torch image to numpy image
    dataset_initial_image = dataset_initial_image.cpu().numpy()
    dataset_initial_image = dataset_initial_image.transpose(1, 2, 0).astype(np.uint8)



    for i in range(episode_start_idx, episode_to_idx):
        action = dataset[i]["action"].cpu().numpy()
        logger.debug(f"target robot pose = {robot_pose}")
        logger.debug(f"current robot pose = {env.get_joint_configuration()}")
        logger.debug(f"current state observation = {obs['state']}")



In [5]:
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# Example structure:
# dataset[i] = {
#     "scene_image": np.array(...),
#     "wrist_wilson_image": np.array(...),
#     "wrist_sophie_image": np.array(...),
#     "action": float
# }

# Extract all actions for the action plot
actions = [entry["action"][0] for entry in dataset]
n = len(dataset)

# --- Create the figure layout ---
fig = plt.figure(figsize=(12, 8))

# Image subplots
ax_scene = fig.add_subplot(2, 3, 1)
ax_wilson = fig.add_subplot(2, 3, 2)
ax_sophie = fig.add_subplot(2, 3, 3)

# Action plot
ax_action = fig.add_subplot(2, 1, 2)

# Display initial images
i0 = 0
im_scene = ax_scene.imshow(dataset[i0]["scene_image"])
ax_scene.set_title("Scene Image")
ax_scene.axis("off")

im_wilson = ax_wilson.imshow(dataset[i0]["wrist_wilson_image"])
ax_wilson.set_title("Wrist Wilson Image")
ax_wilson.axis("off")

im_sophie = ax_sophie.imshow(dataset[i0]["wrist_sophie_image"])
ax_sophie.set_title("Wrist Sophie Image")
ax_sophie.axis("off")

# Display action plot
ax_action.plot(range(n), actions, label="Action")
vline = ax_action.axvline(i0, color="r", linestyle="--", label="Current index")
ax_action.set_xlabel("Index i")
ax_action.set_ylabel("Action value")
ax_action.legend()

# Adjust layout for slider
plt.subplots_adjust(bottom=0.15)

# Slider axis: [left, bottom, width, height]
ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])
slider = Slider(ax_slider, 'Index', 0, n-1, valinit=i0, valstep=1)

# --- Update function ---
def update(val):
    i = int(slider.val)
    im_scene.set_data(dataset[i]["scene_image"])
    im_wilson.set_data(dataset[i]["wrist_wilson_image"])
    im_sophie.set_data(dataset[i]["wrist_sophie_image"])
    vline.set_xdata([i, i])
    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()


KeyboardInterrupt: 